# DR9 Local Overdensity Plotting

This notebook is plotting-only. It reads the saved cluster table with DR9 local-overdensity columns and `lambda_true`/`lambda_spec`, then compares local environment statistics against both spectroscopic richness and redMaPPer richness. It does not rerun the sweep-file counting.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from scipy.stats import binned_statistic, pearsonr, spearmanr

# Resolve the repo root whether this is run from the repo, local_overdensity/,
# or the parent Codex workspace.
repo_root = Path.cwd()
if repo_root.name == 'local_overdensity':
    repo_root = repo_root.parent
elif (repo_root / 'DESI_c3clusters').exists():
    repo_root = repo_root / 'DESI_c3clusters'

OUTPUT_DIR = repo_root / 'local_overdensity' / 'dr9_one_file_outputs'
INPUT_TABLE = OUTPUT_DIR / 'rm_dr9_local_overdensity_sweep_with_lambda_spec.fits'
PLOT_DIR = OUTPUT_DIR / 'local_overdensity_plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

COVERAGE_MIN = 0.8
NBINS = 8
Z_BINS = [(0.1, 0.2), (0.2, 0.3), (0.3, 0.4)]
Z_COLORS = ['crimson', 'darkorange', 'royalblue']

plt.rcParams.update({
    'figure.constrained_layout.use': True,
    'font.size': 13,
    'axes.linewidth': 1.2,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
})

print('Input:', INPUT_TABLE)
print('Plot dir:', PLOT_DIR)


## Load Table And Build Derived Columns

The saved table already contains coverage-corrected densities and the preferred local excess estimator. This cell adds convenience columns for full-annulus-equivalent coverage-corrected counts:

\[
N_{m signal,corr} = rac{N_{m signal}}{f_{m cov,signal}},
\]

\[
N_{m bg,corr} = rac{N_{m bg}}{f_{m cov,bg}}.
\]

For densities, the better correction is already to divide by the covered area:

\[
\Sigma = rac{N}{A f_{m cov}}.
\]

In [ ]:
table = Table.read(INPUT_TABLE)
print(f'Rows: {len(table):,}')
print('Available richness columns:', [c for c in ['LAMBDA', 'lambda_true', 'lambda_spec'] if c in table.colnames])


def col_float(tab, col):
    arr = np.ma.asarray(tab[col], dtype=float)
    return np.ma.filled(arr, np.nan)


# Prefer lambda_true naming, but keep lambda_spec as an alias if needed.
if 'lambda_true' not in table.colnames and 'lambda_spec' in table.colnames:
    table['lambda_true'] = table['lambda_spec']
if 'lambda_spec' not in table.colnames and 'lambda_true' in table.colnames:
    table['lambda_spec'] = table['lambda_true']

coverage_signal = col_float(table, 'coverage_signal_sweep')
coverage_bg = col_float(table, 'coverage_bg_sweep')

N_signal = col_float(table, 'Ngal_signal_DR9_annulus')
N_bg = col_float(table, 'Ngal_bg_DR9_annulus')

# Full-annulus-equivalent corrected counts. These are useful for plotting counts,
# but use the density/excess estimators below for the main science comparison.
table['Ngal_signal_DR9_annulus_covcorr'] = N_signal / coverage_signal
table['Ngal_bg_DR9_annulus_covcorr'] = N_bg / coverage_bg

# Recompute the preferred local excess from covered-area densities, in case the
# table was produced by an older version of the sweep notebook.
area_signal = col_float(table, 'area_signal_deg2')
covered_area_signal = col_float(table, 'covered_area_signal_deg2')
covered_area_bg = col_float(table, 'covered_area_bg_deg2')
Sigma_signal = N_signal / covered_area_signal
Sigma_bg = N_bg / covered_area_bg
Sigma_excess = Sigma_signal - Sigma_bg
Nexcess_local = Sigma_excess * area_signal

table['Sigma_signal_covcorr_recomputed'] = Sigma_signal
table['Sigma_bg_covcorr_recomputed'] = Sigma_bg
table['Sigma_excess_local_recomputed'] = Sigma_excess
table['Nexcess_local_recomputed'] = Nexcess_local

base_mask = (
    np.isfinite(col_float(table, 'lambda_true'))
    & (col_float(table, 'lambda_true') > 0)
    & np.isfinite(col_float(table, 'LAMBDA'))
    & (col_float(table, 'LAMBDA') > 0)
    & np.isfinite(coverage_signal)
    & np.isfinite(coverage_bg)
    & (coverage_signal > COVERAGE_MIN)
    & (coverage_bg > COVERAGE_MIN)
)

print(f'Clusters after richness + coverage cut: {np.count_nonzero(base_mask):,}')


## Plotting Helpers

In [ ]:
def add_binned_mean(ax, x, y, nbins=NBINS, color='crimson', label='binned mean'):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & (x > 0)
    x = x[mask]
    y = y[mask]
    if len(x) < nbins:
        return None

    bins = np.logspace(np.log10(np.nanmin(x)), np.log10(np.nanmax(x)), nbins + 1)
    centers = np.sqrt(bins[:-1] * bins[1:])
    mean, _, _ = binned_statistic(x, y, statistic='mean', bins=bins)
    std, _, _ = binned_statistic(x, y, statistic='std', bins=bins)
    count, _, _ = binned_statistic(x, y, statistic='count', bins=bins)
    sem = std / np.sqrt(np.clip(count, 1, None))
    good = count > 3

    ax.errorbar(
        centers[good],
        mean[good],
        yerr=sem[good],
        fmt='o',
        color=color,
        ecolor=color,
        capsize=3,
        label=label,
        zorder=4,
    )
    return centers, mean, sem, count


def correlation_summary(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & (x > 0)
    if np.count_nonzero(mask) < 3:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 0
    pearson_r, pearson_p = pearsonr(x[mask], y[mask])
    pearson_logx_r, pearson_logx_p = pearsonr(np.log10(x[mask]), y[mask])
    spearman_r, spearman_p = spearmanr(x[mask], y[mask])
    return pearson_r, pearson_p, pearson_logx_r, pearson_logx_p, spearman_r, spearman_p, np.count_nonzero(mask)


def make_scatter_grid(x_col, x_label, filename_prefix):
    x = col_float(table, x_col)
    z = col_float(table, 'Z_SPEC_x')

    y_columns = [
        ('Ngal_signal_DR9_annulus_covcorr', r'$N_{\rm signal}/f_{\rm cov}$'),
        ('Ngal_bg_DR9_annulus_covcorr', r'$N_{\rm bg}/f_{\rm cov}$'),
        ('Sigma_signal_covcorr_recomputed', r'$\Sigma_{\rm signal}$'),
        ('Sigma_bg_covcorr_recomputed', r'$\Sigma_{\rm bg}$'),
        ('Sigma_excess_local_recomputed', r'$\Sigma_{\rm excess,local}$'),
        ('Nexcess_local_recomputed', r'$N_{\rm excess,local}$'),
    ]

    rows = []
    fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)
    axes = axes.ravel()

    for ax, (y_col, y_label) in zip(axes, y_columns):
        y = col_float(table, y_col)
        mask = base_mask & np.isfinite(x) & (x > 0) & np.isfinite(y) & np.isfinite(z)

        sc = ax.scatter(
            x[mask],
            y[mask],
            c=z[mask],
            s=14,
            alpha=0.55,
            cmap='viridis',
            edgecolor='none',
        )
        add_binned_mean(ax, x[mask], y[mask])

        if 'excess' in y_col.lower():
            ax.axhline(0.0, color='black', lw=1.0, alpha=0.55)

        ax.set_xscale('log')
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        ax.legend(frameon=False, fontsize=9)

        pr, pp, plr, plp, sr, sp, n = correlation_summary(x[mask], y[mask])
        rows.append((x_col, y_col, n, pr, pp, plr, plp, sr, sp))

    fig.suptitle(f'Local overdensity statistics versus {x_label}; coverage > {COVERAGE_MIN}', fontsize=15)
    cbar = fig.colorbar(sc, ax=axes.tolist(), pad=0.01)
    cbar.set_label(r'$z_{\rm spec,BCG}$')
    fig.savefig(PLOT_DIR / f'{filename_prefix}_scatter_grid.png', dpi=180)
    plt.show()

    return Table(
        rows=rows,
        names=[
            'x_quantity',
            'y_quantity',
            'N',
            'pearson_r_x',
            'pearson_p_x',
            'pearson_r_logx',
            'pearson_p_logx',
            'spearman_r_x',
            'spearman_p_x',
        ],
    )


def make_focused_plot(x_col, x_label, filename_prefix):
    x = col_float(table, x_col)
    y = col_float(table, 'Nexcess_local_recomputed')
    z = col_float(table, 'Z_SPEC_x')
    mask = base_mask & np.isfinite(x) & (x > 0) & np.isfinite(y) & np.isfinite(z)

    fig, ax = plt.subplots(figsize=(7.2, 5.4))
    sc = ax.scatter(
        x[mask],
        y[mask],
        c=z[mask],
        s=18,
        alpha=0.6,
        cmap='viridis',
        edgecolor='none',
    )
    add_binned_mean(ax, x[mask], y[mask])
    ax.axhline(0.0, color='black', lw=1.0, alpha=0.6)
    ax.set_xscale('log')
    ax.set_xlabel(x_label)
    ax.set_ylabel(r'$N_{\rm excess,local}$')

    pr, pp, plr, plp, sr, sp, n = correlation_summary(x[mask], y[mask])
    ax.text(
        0.04,
        0.96,
        rf'$N={n}$' + '\n' + rf'Spearman $r_s={sr:.2f}$, $p={sp:.1e}$' + '\n' + rf'Pearson $(\log x, y)$ $r={plr:.2f}$, $p={plp:.1e}$',
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=11,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.8),
    )

    ax.legend(frameon=False)
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label(r'$z_{\rm spec,BCG}$')
    fig.savefig(PLOT_DIR / f'{filename_prefix}_Nexcess_local.png', dpi=180)
    plt.show()


## Local Overdensity Versus Spectroscopic Richness

In [ ]:
corr_lambda_true = make_scatter_grid(
    'lambda_true',
    r'$\lambda_{\rm true}$',
    'local_overdensity_vs_lambda_true',
)
make_focused_plot(
    'lambda_true',
    r'$\lambda_{\rm true}$',
    'local_overdensity_vs_lambda_true',
)

corr_lambda_true.write(
    PLOT_DIR / 'local_overdensity_vs_lambda_true_correlations.ecsv',
    format='ascii.ecsv',
    overwrite=True,
)
corr_lambda_true


## Local Overdensity Versus redMaPPer Richness

In [ ]:
corr_lambda_rm = make_scatter_grid(
    'LAMBDA',
    r'$\lambda_{\rm RM}$',
    'local_overdensity_vs_lambda_RM',
)
make_focused_plot(
    'LAMBDA',
    r'$\lambda_{\rm RM}$',
    'local_overdensity_vs_lambda_RM',
)

corr_lambda_rm.write(
    PLOT_DIR / 'local_overdensity_vs_lambda_RM_correlations.ecsv',
    format='ascii.ecsv',
    overwrite=True,
)
corr_lambda_rm


## Redshift-Binned Local Overdensity Trends

The following plots repeat the local-overdensity comparisons in redshift bins: `[0.1,0.2]`, `[0.2,0.3]`, and `[0.3,0.4]`. The same coverage mask is applied within each redshift bin.


In [ ]:
def make_redshift_binned_grid(x_col, x_label, filename_prefix):
    x = col_float(table, x_col)
    z = col_float(table, 'Z_SPEC_x')

    y_columns = [
        ('Ngal_signal_DR9_annulus_covcorr', r'$N_{\rm signal}/f_{\rm cov}$'),
        ('Ngal_bg_DR9_annulus_covcorr', r'$N_{\rm bg}/f_{\rm cov}$'),
        ('Sigma_signal_covcorr_recomputed', r'$\Sigma_{\rm signal}$'),
        ('Sigma_bg_covcorr_recomputed', r'$\Sigma_{\rm bg}$'),
        ('Sigma_excess_local_recomputed', r'$\Sigma_{\rm excess,local}$'),
        ('Nexcess_local_recomputed', r'$N_{\rm excess,local}$'),
    ]

    all_rows = []

    for y_col, y_label in y_columns:
        y = col_float(table, y_col)
        fig, axes = plt.subplots(1, len(Z_BINS), figsize=(16, 4.7), sharex=True, sharey=False)

        for ax, (zlo, zhi), color in zip(axes, Z_BINS, Z_COLORS):
            zmask = (z >= zlo) & (z < zhi)
            mask = base_mask & zmask & np.isfinite(x) & (x > 0) & np.isfinite(y)

            ax.scatter(
                x[mask],
                y[mask],
                s=14,
                alpha=0.45,
                color=color,
                edgecolor='none',
            )
            add_binned_mean(ax, x[mask], y[mask], nbins=NBINS, color='black')

            if 'excess' in y_col.lower():
                ax.axhline(0.0, color='black', lw=1.0, alpha=0.45)

            ax.set_xscale('log')
            ax.set_xlabel(x_label)
            ax.set_title(rf'${zlo}<z<{zhi}$, $N={np.count_nonzero(mask)}$')
            ax.legend(frameon=False, fontsize=9)

            pr, pp, plr, plp, sr, sp, n = correlation_summary(x[mask], y[mask])
            ax.text(
                0.04,
                0.96,
                rf'$r_s={sr:.2f}$' + '\n' + rf'$p={sp:.1e}$',
                transform=ax.transAxes,
                ha='left',
                va='top',
                fontsize=10,
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.75),
            )

            all_rows.append((x_col, y_col, zlo, zhi, n, pr, pp, plr, plp, sr, sp))

        axes[0].set_ylabel(y_label)
        fig.suptitle(f'{y_label} versus {x_label} in redshift bins; coverage > {COVERAGE_MIN}', fontsize=15)
        safe_y_col = y_col.replace('/', '_').replace(' ', '_')
        fig.savefig(PLOT_DIR / f'{filename_prefix}_{safe_y_col}_z_binned.png', dpi=180)
        plt.show()

    return Table(
        rows=all_rows,
        names=[
            'x_quantity',
            'y_quantity',
            'z_low',
            'z_high',
            'N',
            'pearson_r_x',
            'pearson_p_x',
            'pearson_r_logx',
            'pearson_p_logx',
            'spearman_r_x',
            'spearman_p_x',
        ],
    )


def make_redshift_binned_focused_plot(x_col, x_label, filename_prefix):
    x = col_float(table, x_col)
    y = col_float(table, 'Nexcess_local_recomputed')
    z = col_float(table, 'Z_SPEC_x')

    fig, axes = plt.subplots(1, len(Z_BINS), figsize=(16, 4.7), sharex=True, sharey=True)

    for ax, (zlo, zhi), color in zip(axes, Z_BINS, Z_COLORS):
        zmask = (z >= zlo) & (z < zhi)
        mask = base_mask & zmask & np.isfinite(x) & (x > 0) & np.isfinite(y)

        ax.scatter(
            x[mask],
            y[mask],
            s=18,
            alpha=0.55,
            color=color,
            edgecolor='none',
        )
        add_binned_mean(ax, x[mask], y[mask], nbins=NBINS, color='black')
        ax.axhline(0.0, color='black', lw=1.0, alpha=0.55)
        ax.set_xscale('log')
        ax.set_xlabel(x_label)
        ax.set_title(rf'${zlo}<z<{zhi}$, $N={np.count_nonzero(mask)}$')

        pr, pp, plr, plp, sr, sp, n = correlation_summary(x[mask], y[mask])
        ax.text(
            0.04,
            0.96,
            rf'$r_s={sr:.2f}$' + '\n' + rf'$p={sp:.1e}$',
            transform=ax.transAxes,
            ha='left',
            va='top',
            fontsize=10,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.75),
        )
        ax.legend(frameon=False, fontsize=9)

    axes[0].set_ylabel(r'$N_{\rm excess,local}$')
    fig.suptitle(rf'$N_{{\rm excess,local}}$ versus {x_label} in redshift bins; coverage $>{COVERAGE_MIN}$', fontsize=15)
    fig.savefig(PLOT_DIR / f'{filename_prefix}_Nexcess_local_z_binned.png', dpi=180)
    plt.show()


In [ ]:
corr_lambda_true_z = make_redshift_binned_grid(
    'lambda_true',
    r'$\lambda_{\rm true}$',
    'local_overdensity_vs_lambda_true',
)
make_redshift_binned_focused_plot(
    'lambda_true',
    r'$\lambda_{\rm true}$',
    'local_overdensity_vs_lambda_true',
)

corr_lambda_true_z.write(
    PLOT_DIR / 'local_overdensity_vs_lambda_true_redshift_binned_correlations.ecsv',
    format='ascii.ecsv',
    overwrite=True,
)
corr_lambda_true_z


In [ ]:
corr_lambda_rm_z = make_redshift_binned_grid(
    'LAMBDA',
    r'$\lambda_{\rm RM}$',
    'local_overdensity_vs_lambda_RM',
)
make_redshift_binned_focused_plot(
    'LAMBDA',
    r'$\lambda_{\rm RM}$',
    'local_overdensity_vs_lambda_RM',
)

corr_lambda_rm_z.write(
    PLOT_DIR / 'local_overdensity_vs_lambda_RM_redshift_binned_correlations.ecsv',
    format='ascii.ecsv',
    overwrite=True,
)
corr_lambda_rm_z


## Notes On Coverage Correction

For edge effects, dividing a **count** by the coverage fraction is a reasonable full-annulus-equivalent correction if the missing part of the annulus is representative of the observed part:

\[
N_{m corr} = rac{N_{m obs}}{f_{m cov}}.
\]

For a **density**, it is cleaner to divide by the observed/covered area rather than divide a density by coverage afterward:

\[
\Sigma = rac{N_{m obs}}{A_{m annulus} f_{m cov}} = rac{N_{m obs}}{A_{m covered}}.
\]

For the local-background excess, the preferred estimator in this notebook is:

\[
\Sigma_{m excess,local} = rac{N_{m signal}}{A_{m signal} f_{m cov,signal}} - rac{N_{m bg}}{A_{m bg} f_{m cov,bg}},
\]

\[
N_{m excess,local} = \Sigma_{m excess,local} A_{m signal}.
\]

This is already coverage corrected. The coverage cut remains important because dividing by small coverage values can produce very noisy outliers.